# Notebook 1: Exploratory Data Analysis — CAFA6

Files used:
- `data/raw/Train/train_sequences.fasta` — 82,404 UniProt sequences
- `data/raw/Train/train_terms.tsv` — 537,027 GO annotations (EntryID, term, aspect)
- `data/raw/IA.tsv` — per-term information accretion weights
- `data/raw/Test/testsuperset.fasta` — 224,309 test sequences

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from preprocessing import parse_fasta, parse_go_annotations, filter_sequences
from evaluate import load_ia_weights

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

TRAIN_DIR = Path('../data/raw/Train')
TEST_DIR  = Path('../data/raw/Test')
RAW_DIR   = Path('../data/raw')

## 1. Load Data

In [ ]:
sequences = parse_fasta(TRAIN_DIR / 'train_sequences.fasta')
print(f'Total raw sequences: {len(sequences):,}')

sequences_filtered = filter_sequences(sequences, min_len=1, max_len=2048)
print(f'After filtering:    {len(sequences_filtered):,}')

In [ ]:
annotations = parse_go_annotations(TRAIN_DIR / 'train_terms.tsv')
print(annotations.head(10))
print(f'\nTotal annotations: {len(annotations):,}')
print(f'\nPer ontology:')
print(annotations['ontology'].value_counts())

In [ ]:
test_sequences = parse_fasta(TEST_DIR / 'testsuperset.fasta')
print(f'Test sequences: {len(test_sequences):,}')

## 2. Sequence Length Distribution

In [ ]:
lengths = [len(s) for s in sequences_filtered.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(lengths, bins=100, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Sequence Length (aa)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')
axes[0].axvline(np.median(lengths), color='red', linestyle='--',
                label=f'Median={int(np.median(lengths))}')
axes[0].legend()

axes[1].hist(np.log10(lengths), bins=80, color='coral', edgecolor='white')
axes[1].set_xlabel('log10(Sequence Length)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log-Scale Length Distribution')

plt.tight_layout()
Path('../reports').mkdir(exist_ok=True)
plt.savefig('../reports/seq_length_dist.png', dpi=150)
plt.show()

print(f'Min: {min(lengths)} | Max: {max(lengths)} | Median: {int(np.median(lengths))} | Mean: {np.mean(lengths):.0f}')
for pct in [50, 90, 95, 99]:
    print(f'  {pct}th percentile: {int(np.percentile(lengths, pct))} aa')

## 3. GO Term Frequency per Ontology

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ont in zip(axes, ['BPO', 'MFO', 'CCO']):
    sub = annotations[annotations['ontology'] == ont]
    term_counts = sub.groupby('go_term')['protein_id'].nunique().sort_values(ascending=False)

    ax.hist(term_counts.values, bins=60, log=True, color='mediumseagreen', edgecolor='white')
    ax.set_xlabel('Proteins per GO Term')
    ax.set_ylabel('Count (log scale)')
    ax.set_title(f'{ont} — {len(term_counts):,} unique terms')

    for cutoff in [10, 50, 100]:
        n = (term_counts >= cutoff).sum()
        print(f'{ont}: {n:,} terms with ≥{cutoff} proteins')

plt.tight_layout()
plt.savefig('../reports/go_term_frequency.png', dpi=150)
plt.show()

## 4. Labels per Protein

In [ ]:
annotated_pids = set(sequences_filtered.keys()) & set(annotations['protein_id'])
print(f'Proteins with sequence AND annotation: {len(annotated_pids):,}')

subset = annotations[annotations['protein_id'].isin(annotated_pids)]
terms_per_protein = subset.groupby('protein_id')['go_term'].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(terms_per_protein.values, bins=80, color='mediumpurple', edgecolor='white')
ax.set_xlabel('GO Terms per Protein (all ontologies)')
ax.set_ylabel('Count')
ax.set_title('Label Cardinality')
ax.axvline(terms_per_protein.median(), color='red', linestyle='--',
           label=f'Median={terms_per_protein.median():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/terms_per_protein.png', dpi=150)
plt.show()

## 5. Information Accretion (IA) Distribution

In [ ]:
ia_df = pd.read_csv(RAW_DIR / 'IA.tsv', sep='\t', header=None, names=['go_term', 'ia'])
print(f'IA values for {len(ia_df):,} GO terms')
print(ia_df.describe())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ia_df['ia'], bins=80, color='salmon', edgecolor='white')
ax.set_xlabel('Information Accretion (IA)')
ax.set_ylabel('Count')
ax.set_title('IA Distribution — used for Smin metric')
plt.tight_layout()
plt.savefig('../reports/ia_distribution.png', dpi=150)
plt.show()

## 6. Top 20 Most Frequent GO Terms per Ontology

In [ ]:
for ont in ['BPO', 'MFO', 'CCO']:
    sub = annotations[annotations['ontology'] == ont]
    top20 = sub.groupby('go_term')['protein_id'].nunique().nlargest(20)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue')
    ax.set_xlabel('Number of Proteins')
    ax.set_title(f'Top 20 GO Terms — {ont}')
    plt.tight_layout()
    plt.savefig(f'../reports/top20_{ont.lower()}.png', dpi=150)
    plt.show()

## 7. Summary

In [ ]:
print('=== CAFA6 Dataset Summary ===')
print(f'Train sequences         : {len(sequences):,}')
print(f'After length filter     : {len(sequences_filtered):,}')
print(f'Proteins with annotation: {len(annotated_pids):,}')
print(f'Total annotations       : {len(annotations):,}')
print(f'Unique GO terms         : {annotations["go_term"].nunique():,}')
print(f'Test sequences          : {len(test_sequences):,}')
print(f'IA-weighted terms       : {len(ia_df):,}')
print()
for ont in ['BPO', 'MFO', 'CCO']:
    sub = annotations[annotations['ontology'] == ont]
    print(f'{ont}: {sub["go_term"].nunique():,} unique terms | {sub["protein_id"].nunique():,} annotated proteins')